# 5. Combining Profiles

## Purpose
Concatenate all per-well-FOV parquet files for a single patient into three
patient-level combined parquets (SC, organoid, nucleocentric).

This is **step 5 of Stage 4 (image-based profiling)**. It runs once per patient
and is typically submitted as a per-patient SLURM job.

## Inputs
- `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`
  - `sc_profiles_{well_fov}_related.parquet`
  - `organoid_profiles_{well_fov}_related.parquet`
  - `nucleocentric_profiles_{well_fov}_related.parquet`

## Outputs
Three combined parquets in `data/{patient}/image_based_profiles/2.combined_profiles/`:

| File | Content |
|---|---|
| `sc.parquet` | All SC profiles stacked across FOVs |
| `organoid.parquet` | All organoid profiles stacked across FOVs |
| `nucleocentric.parquet` | All nucleocentric profiles stacked across FOVs |

## Notes
- Concatenation uses DuckDB `union_by_name=true`, which aligns columns by name
  rather than position. FOVs with missing columns (e.g. empty scaffold tables)
  will have those columns filled with NULL.
- Brightfield (BF) channel features are removed after concatenation as they are
  not part of the fluorescent cell painting panel and are not used in profiling.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0030_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
# set paths
profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles"
).resolve(strict=True)
# output_paths
sc_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/sc.parquet"
).resolve()
organoid_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/organoid.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/nucleocentric.parquet"
).resolve()
organoid_merged_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Discover all per-FOV parquet files under 1.related_profiles/.
# The directory structure is 1.related_profiles/{well_fov}/*.parquet,
# so one wildcard level is sufficient.
profiles = list(profiles_path.rglob("*/*.parquet"))

In [5]:
# Split files by profile type using filename prefix.
# Expected prefixes: 'sc_', 'organoid_', 'nucleocentric_'.
sc_profiles = [str(x) for x in profiles if x.name.startswith("sc_")]
organoid_profiles = [str(x) for x in profiles if x.name.startswith("organoid_")]
nucleocentric_profiles = [
    str(x) for x in profiles if x.name.startswith("nucleocentric_")
]

In [6]:
for x in nucleocentric_profiles:
    df = pd.read_parquet(x)
    if df.isnull().any().any():
        print(f"Null values found in {x}")
df

Null values found in /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0030_T1/image_based_profiles/1.related_profiles/C10-4/nucleocentric_profiles_C10-4_related.parquet
Null values found in /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0030_T1/image_based_profiles/1.related_profiles/C7-2/nucleocentric_profiles_C7-2_related.parquet
Null values found in /home/lippincm/Documents/NF1_3D_organoid_profiling_pipeline/data/NF0030_T1/image_based_profiles/1.related_profiles/D3-1/nucleocentric_profiles_D3-1_related.parquet


,object_id,image_set,Nucleocentric_Mito_CHAMMI75_Feature0,Nucleocentric_Mito_CHAMMI75_Feature1,Nucleocentric_Mito_CHAMMI75_Feature10,Nucleocentric_Mito_CHAMMI75_Feature100,Nucleocentric_Mito_CHAMMI75_Feature101,Nucleocentric_Mito_CHAMMI75_Feature102,Nucleocentric_Mito_CHAMMI75_Feature103,Nucleocentric_Mito_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99,ParentOrganoid
0,1,D10-3,-4.698085,-3.109749,2.937747,0.794603,-0.486432,0.545599,3.998104,3.206762,...,-0.145654,0.041816,-0.010768,0.057913,-0.007872,-0.081265,0.169721,0.288596,0.214424,1
1,2,D10-3,-2.876151,-1.948808,4.234421,-0.435982,-1.268100,0.762916,3.593872,5.431098,...,-0.079379,0.094818,-0.011266,0.017841,-0.045267,-0.028417,0.199428,0.352314,0.193595,1
2,3,D10-3,-3.429457,-5.873388,4.754021,1.303932,-0.678502,-2.598720,4.040520,1.727761,...,-0.078445,0.047159,-0.010648,0.030756,-0.024158,-0.090093,0.200203,0.348180,0.217930,1
3,4,D10-3,-3.866829,-3.123580,4.576599,-0.269965,-2.118972,-0.516158,-0.544355,2.628209,...,-0.049047,0.042644,-0.010749,0.028052,-0.010946,-0.057543,0.218869,0.346184,0.206010,1
4,5,D10-3,-3.532081,-2.770919,0.908017,1.030769,-0.667008,1.765043,6.577331,1.905848,...,-0.070700,0.015185,-0.010589,0.009773,-0.004052,-0.052950,0.205564,0.395356,0.206077,1
5,6,D10-3,-5.285849,-3.369694,6.311458,1.374958,-1.744378,1.153503,2.947392,1.136238,...,-0.089219,0.031743,-0.010713,-0.003857,0.023830,-0.011939,0.209430,0.326508,0.177111,1
6,7,D10-3,-4.061612,-1.776379,3.803882,0.249281,0.063960,-0.363364,2.591492,3.470561,...,-0.058607,0.094622,-0.010648,0.026236,0.030064,-0.021361,0.245787,0.317055,0.234534,-1
7,8,D10-3,-2.713203,-0.265863,0.545385,1.053423,2.046300,4.264626,6.855599,1.027850,...,0.057481,0.113536,-0.010957,0.043413,-0.052365,-0.023580,0.138033,0.272923,0.309261,1
8,9,D10-3,-3.547326,-1.302578,-0.091603,1.141024,1.196989,2.580527,5.699528,-0.599255,...,-0.095477,0.056875,-0.010721,0.023420,-0.029690,-0.067387,0.257887,0.374860,0.246029,1
9,10,D10-3,-2.583375,-1.480098,2.342122,1.687453,-0.626710,1.465380,5.219787,3.825478,...,-0.058535,0.071968,-0.010705,0.049913,0.003853,0.027307,0.219241,0.309030,0.213047,1


In [7]:
# Concatenate per-FOV parquets for each profile type using DuckDB.
# union_by_name=true aligns columns by name rather than position, so FOVs with
# differing column sets (e.g. empty scaffold tables from notebook 1) are handled
# gracefully — missing columns are filled with NULL rather than causing an error.

with duckdb.connect() as conn:
    sc_profile = conn.execute(
        f"SELECT * FROM read_parquet({sc_profiles}, union_by_name=true)"
    ).df()
    organoid_profile = conn.execute(
        f"SELECT * FROM read_parquet({organoid_profiles}, union_by_name=true)"
    ).df()
    nucleocentric_profile = conn.execute(
        f"SELECT * FROM read_parquet({nucleocentric_profiles}, union_by_name=true)"
    ).df()

print(f"Single-cell profiles concatenated. Shape: {sc_profile.shape}")
print(f"Organoid profiles concatenated. Shape: {organoid_profile.shape}")
print(f"Nucleocentric profiles concatenated. Shape: {nucleocentric_profile.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Single-cell profiles concatenated. Shape: (3832, 11889)
Organoid profiles concatenated. Shape: (937, 3962)
Nucleocentric profiles concatenated. Shape: (3835, 3075)


## Remove all BF channels


In [8]:
# Remove brightfield (BF) channel features from all three profile types.
# BF is a transmitted-light channel not part of the fluorescent cell painting
# panel; its features are may not meaningful for morphological profiling.
# and are interpreted differently
# Note: if no BF columns exist in the data, these drops are no-ops.

bf_cols_sc = [col for col in sc_profile.columns if "BF" in col]
sc_profile = sc_profile.drop(columns=bf_cols_sc)
print(f"SC: dropped {len(bf_cols_sc)} BF columns. Shape: {sc_profile.shape}")

bf_cols_organoid = [col for col in organoid_profile.columns if "BF" in col]
organoid_profile = organoid_profile.drop(columns=bf_cols_organoid)
print(
    f"Organoid: dropped {len(bf_cols_organoid)} BF columns. Shape: {organoid_profile.shape}"
)

bf_cols_nucleocentric = [col for col in nucleocentric_profile.columns if "BF" in col]
nucleocentric_profile = nucleocentric_profile.drop(columns=bf_cols_nucleocentric)
print(
    f"Nucleocentric: dropped {len(bf_cols_nucleocentric)} BF columns. Shape: {nucleocentric_profile.shape}"
)

SC: dropped 0 BF columns. Shape: (3832, 11889)
Organoid: dropped 0 BF columns. Shape: (937, 3962)
Nucleocentric: dropped 0 BF columns. Shape: (3835, 3075)


In [9]:
sc_profile.to_parquet(sc_merged_output_path, index=False)
organoid_profile.to_parquet(organoid_merged_output_path, index=False)
nucleocentric_profile.to_parquet(nucleocentric_profile_output_path, index=False)

In [10]:
nucleocentric_profile

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,1,F6-4,-0.497709,-0.142490,0.175844,-0.016173,-0.013257,0.104504,-0.062733,-0.064247,...,4.183833,-8.172391,-0.948838,3.036109,-4.607002,-1.952724,4.874544,-0.656801,-1.657164,-1.0
1,2,F6-4,-0.210915,-0.263298,0.149069,-0.009817,-0.071562,0.248182,0.054136,-0.140743,...,5.682166,-5.185552,-1.278293,4.949970,-2.048710,-3.476991,2.552736,-2.169322,-4.140951,-1.0
2,3,F6-4,-0.269594,-0.167702,0.143264,-0.092674,-0.013394,0.085267,0.082247,-0.211441,...,0.326889,-9.549928,-1.545778,5.846510,-1.858824,-1.795626,4.165531,-5.072268,-3.248634,-1.0
3,4,F6-4,-0.323487,-0.206845,0.124149,0.010363,-0.021326,0.201515,0.012824,-0.146776,...,4.059972,-6.487916,-0.075907,0.427790,-3.112464,-3.292485,4.867661,0.904406,-3.279779,-1.0
4,5,F6-4,0.017735,-0.116598,0.340518,0.119946,-0.102427,0.281943,0.208962,-0.142238,...,3.489267,-9.044875,-0.562577,1.090262,-5.108995,-3.456485,3.141604,-1.361062,-2.699798,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3830,14,D10-3,0.021332,-0.157666,0.137499,-0.006111,-0.006147,0.256801,0.188680,-0.133158,...,1.936190,-5.112000,-0.294041,4.451430,-4.973634,-3.365321,5.447654,-2.948100,-0.449937,1.0
3831,15,D10-3,-0.310072,-0.174222,0.178813,-0.002651,-0.005952,0.174146,0.053305,-0.176924,...,1.955117,-8.332478,-3.605325,2.797621,-7.950428,-1.280566,3.500873,-1.585945,-1.268700,1.0
3832,16,D10-3,-0.046022,-0.149100,0.041746,-0.043122,-0.040170,0.376804,0.007777,-0.108147,...,1.276794,-3.647711,-2.746424,5.877475,-5.256845,-2.720346,2.307742,-3.492917,-2.364886,1.0
3833,17,D10-3,0.004818,-0.143801,0.164994,-0.067205,0.036325,0.085499,0.280330,-0.128751,...,-0.064266,-3.559561,-3.274412,2.588868,-4.950966,-2.709081,3.624469,-2.512587,-0.092043,1.0
